In [ ]:
# (DO NOT containerize this cell)
# Data handling parameters
param_minio_endpoint = 'scruffy.lab.uvalight.net:9000'
param_minio_virtual_lab_bucket = 'naa-vre-laserfarm'
param_max_nodes = 2

In [ ]:
# Secrets (DO NOT containerize this cell)
from SecretsProvider import SecretsProvider
from getpass import getpass

secrets_provider = SecretsProvider(input_func=getpass)
secret_minio_access_key = secrets_provider.get_secret('secret_minio_access_key')
secret_minio_secret_key = secrets_provider.get_secret('secret_minio_secret_key')

In [ ]:
# (DO NOT containerize this cell)
from minio import Minio
from pathlib import Path
from typing import List
import math
import json
import io
import yaml
import random

In [ ]:
# (DO NOT containerize this cell)
# Dummy provide input
folder_batch_from = "testOriginal"
folder_to_batch_to = "toRetile"

In [ ]:
# Create file batches
def save_lists_to_cloud_storage(batches: int, batch_folder: str, file_folder: str, file_basename: str, filenames: List[str]) -> List[str]:
    random.shuffle(filenames)
    filename_batches = split_into_batches(batches, filenames)
    batch_filenames = []
    for n, batch in enumerate(filename_batches):
        filepath = f"{batch_folder}/{n}.{file_basename}"
        print(f"storing batch: [{batch[0]} ... n={len(batch)}] as json in {filepath}")
        file_information = {
            "folder": file_folder,
            "files": batch
        }
        json_batch = json.dumps(file_information)
        save_json_in_cloud_storage(filepath, json_batch)
        batch_filenames.append(filepath)
    return batch_filenames

def split_into_batches(max_batches: int, filenames: List[str]):
    batch_size = math.ceil(len(filenames)/max_batches)
    return[filenames[i:i + batch_size] for i in range(0, len(filenames), batch_size)]
    
def save_json_in_cloud_storage(filepath: str, json) -> str:
    json_file = io.BytesIO(json.encode("utf-8"))
    minio_client.put_object(bucket_name=param_minio_virtual_lab_bucket, object_name=str(filepath), data=json_file, length=len(json), content_type="application/json")

minio_client = Minio(
    param_minio_endpoint, 
    access_key=secret_minio_access_key,
    secret_key=secret_minio_secret_key,
    secure=True
)

objects = minio_client.list_objects(param_minio_virtual_lab_bucket, prefix=folder_batch_from, recursive=True)
filenames = [str(Path(obj.object_name).name) for obj in objects]
workflow_files: List[str] = save_lists_to_cloud_storage(batches=2, batch_folder=folder_to_batch_to, file_folder=folder_batch_from, file_basename=f"{folder_to_batch_to}.yaml", filenames=filenames)

In [ ]:
# (DO NOT containerize this cell)
# Dummy use output
minio_client = Minio(
    param_minio_endpoint, 
    access_key=secret_minio_access_key,
    secret_key=secret_minio_secret_key,
    secure=True
)

file_information = yaml.safe_load(minio_client.get_object(bucket_name=param_minio_virtual_lab_bucket, object_name=workflow_files[0]).read().decode("utf-8"))
print(file_information)